In [4]:
# --- Zero-shot with GPU, Dataset batching, torchvision stub, + DISTRACTORS
#     Outputs: z_nature, z_society, z_culture,
#              z_greenwashing_discourse, z_greenwashing_claims,
#              z_transformative_discourse, z_transformative_action,
#              z_greenwashing (alias=claims), z_transformativeness (alias=action)

import os, sys, csv, warnings, time, types, importlib.machinery, pathlib
import pandas as pd
warnings.filterwarnings("ignore", category=UserWarning)

# ========= Paths & caches =========
BASE_DIR = "/projappl/project_2004147/visions/bertopic_with_zeroshot_v9"
IN_CSV   = os.path.join(BASE_DIR, "mocked_dataset_generated_by_chatgpt.csv")
OUT_CSV  = os.path.join(BASE_DIR, "nff_transf_green_zeroshot_anon.csv")

HF_CACHE = "/scratch/project_2004147/visions/hf_cache"
os.makedirs(HF_CACHE, exist_ok=True)
os.environ["HF_HOME"] = HF_CACHE
os.environ["HUGGINGFACE_HUB_CACHE"] = HF_CACHE
os.environ["TRANSFORMERS_CACHE"] = HF_CACHE
os.environ["XDG_CACHE_HOME"] = HF_CACHE
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["TRANSFORMERS_NO_TORCHVISION"] = "1"
os.environ["PYTHONNOUSERSITE"] = "1"

SITEPKGS = "/scratch/project_2004147/visions/sitepkgs"
if SITEPKGS not in sys.path:
    sys.path.insert(0, SITEPKGS)
sys.path = [p for p in sys.path if "venv_zeroshot/lib/python3.10/site-packages" not in p]

# ========= TorchVision stub WITH valid __spec__ =========
for m in list(sys.modules.keys()):
    if m == "torchvision" or m.startswith("torchvision."):
        del sys.modules[m]

_tv = types.ModuleType("torchvision")
_tv.__version__ = "0.0-stub"
_tv.__file__ = "<stub>"
_tv.__path__ = []
_tv.__spec__ = importlib.machinery.ModuleSpec(name="torchvision", loader=None, is_package=True)

_transforms = types.ModuleType("torchvision.transforms")
_transforms.__file__ = "<stub>"
_transforms.__spec__ = importlib.machinery.ModuleSpec(name="torchvision.transforms", loader=None, is_package=False)
class _InterpolationMode:
    NEAREST = BOX = BILINEAR = BICUBIC = LANCZOS = HAMMING = None
_transforms.InterpolationMode = _InterpolationMode()

_tv.transforms = _transforms
sys.modules["torchvision"] = _tv
sys.modules["torchvision.transforms"] = _transforms

# ========= Labels =========
# NFF triad (concise, discriminative)
NATURE_LABEL         = "nature, wildlife, biodiversity, ecosystems, and landscapes"
SOCIETY_LABEL        = "green technology, business, industry, and natural resources"
CULTURE_LABEL        = "people, community values, and cultural heritage"

# Greenwashing & Transformative — discourse vs claims/action
GW_DISCOURSE_LABEL   = "talking about greenwashing"
GW_CLAIMS_LABEL      = "promotional eco claim, green marketing"
TR_DISCOURSE_LABEL   = "talking about transformative change"
TR_ACTION_LABEL      = "implemented environmental action"

# Primary labels we will export
PRIMARY_LABELS = [
    NATURE_LABEL, SOCIETY_LABEL, CULTURE_LABEL,
    GW_DISCOURSE_LABEL, GW_CLAIMS_LABEL,
    TR_DISCOURSE_LABEL, TR_ACTION_LABEL
]

# Lightweight distractors to sharpen decision boundaries
DISTRACTORS = [
    "general climate politics",
    "technology and business news",
    "ordinary sustainability news",
    "company financial update",
    "sports or entertainment"
]

CANDIDATE_LABELS = PRIMARY_LABELS + DISTRACTORS

COLUMN_MAP = {
    NATURE_LABEL:         "z_nature",
    SOCIETY_LABEL:        "z_society",
    CULTURE_LABEL:        "z_culture",
    GW_DISCOURSE_LABEL:   "z_greenwashing_discourse",
    GW_CLAIMS_LABEL:      "z_greenwashing_claims",
    TR_DISCOURSE_LABEL:   "z_transformative_discourse",
    TR_ACTION_LABEL:      "z_transformative_action",
}

HYPOTHESIS = "This text is about {}."
MODEL_ID   = "facebook/bart-large-mnli"

# ========= Load CSV & pick text =========
def pick_text(df: pd.DataFrame) -> pd.Series:
    if "text" in df.columns:
        base = df["text"].astype(str)
        use  = base if "text_clean" not in df.columns else base.where(base.str.strip().ne(""), df["text_clean"].astype(str))
    elif "text_clean" in df.columns:
        use = df["text_clean"].astype(str)
    else:
        raise KeyError("Neither 'text' nor 'text_clean' found.")
    use = (use.str.replace(r"http\S+|www\.\S+", " ", regex=True)
              .str.replace(r"(?:^|\s)@[\w_]+", " ", regex=True)
              .str.replace(r"#", " ", regex=True)
              .str.replace(r"\s+", " ", regex=True)
              .str.strip())
    return use.fillna("")

df = pd.read_csv(IN_CSV, low_memory=False)
texts = pick_text(df)
n = len(texts)
print(f"Loaded {n:,} rows from {IN_CSV}", flush=True)

# ========= Model / pipeline (GPU FP16 if available) =========
import torch
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification

use_gpu = torch.cuda.is_available()
device  = 0 if use_gpu else -1
print("GPU available:", use_gpu, flush=True)
if use_gpu:
    try:
        print("Using GPU:", torch.cuda.get_device_name(0), flush=True)
    except Exception:
        pass

tok = AutoTokenizer.from_pretrained(MODEL_ID, cache_dir=HF_CACHE)
mdl = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID, cache_dir=HF_CACHE,
    torch_dtype=(torch.float16 if use_gpu else torch.float32),
)
if use_gpu:
    mdl = mdl.to(0)

clf = pipeline(
    "zero-shot-classification",
    model=mdl,
    tokenizer=tok,
    device=device,
    torch_dtype=(torch.float16 if use_gpu else None),
)

# ========= Batch size heuristic =========
if use_gpu:
    try:
        vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    except Exception:
        vram_gb = 16
    if vram_gb >= 40:   BATCH_SIZE = 256
    elif vram_gb >= 24: BATCH_SIZE = 192
    elif vram_gb >= 16: BATCH_SIZE = 128
    else:               BATCH_SIZE = 96
else:
    BATCH_SIZE = 64
print("BATCH_SIZE =", BATCH_SIZE, flush=True)

# ========= Try Dataset-based batching; else fallback =========
use_datasets = False
try:
    from datasets import Dataset
    ds = Dataset.from_pandas(pd.DataFrame({"text": texts}))
    use_datasets = True
    print("HuggingFace 'datasets' available: using Dataset-based batching.", flush=True)
except Exception:
    print("HuggingFace 'datasets' not available -> using manual batching.", flush=True)
    ds = None

# ========= Inference =========
z_cols = {col_name: [] for col_name in COLUMN_MAP.values()}
start_time = time.time()
total_batches = (n + BATCH_SIZE - 1) // BATCH_SIZE
PRINT_EVERY = 200

if use_datasets:
    for bi, start_idx in enumerate(range(0, n, BATCH_SIZE), start=1):
        end_idx = min(start_idx + BATCH_SIZE, n)
        sub = ds.select(range(start_idx, end_idx))
        out = clf(
            sub["text"],
            candidate_labels=CANDIDATE_LABELS,
            hypothesis_template=HYPOTHESIS,
            multi_label=True,
            batch_size=min(BATCH_SIZE, end_idx - start_idx),
            truncation=True,
            padding=True,
        )
        if isinstance(out, dict):
            out = [out]
        for res in out:
            smap = dict(zip(res["labels"], res["scores"]))
            for lab, col in COLUMN_MAP.items():
                z_cols[col].append(float(smap.get(lab, 0.0)))
        if (bi % PRINT_EVERY == 0) or (bi == total_batches):
            elapsed = time.time() - start_time
            rate = bi / elapsed if elapsed > 0 else float("inf")
            eta = (total_batches - bi) / rate if rate > 0 else float("inf")
            print(f"[{bi}/{total_batches}] {bi/total_batches:6.2%} | {elapsed/60:.1f} min elapsed | ETA ~{eta/60:.1f} min", flush=True)
else:
    for bi, start_idx in enumerate(range(0, n, BATCH_SIZE), start=1):
        batch = texts.iloc[start_idx:start_idx + BATCH_SIZE].tolist()
        out = clf(
            batch,
            candidate_labels=CANDIDATE_LABELS,
            hypothesis_template=HYPOTHESIS,
            multi_label=True,
            batch_size=min(BATCH_SIZE, len(batch)),
            truncation=True,
            padding=True,
        )
        if isinstance(out, dict):
            out = [out]
        for res in out:
            smap = dict(zip(res["labels"], res["scores"]))
            for lab, col in COLUMN_MAP.items():
                z_cols[col].append(float(smap.get(lab, 0.0)))
        if (bi % PRINT_EVERY == 0) or (bi == total_batches):
            elapsed = time.time() - start_time
            rate = bi / elapsed if elapsed > 0 else float("inf")
            eta = (total_batches - bi) / rate if rate > 0 else float("inf")
            print(f"[{bi}/{total_batches}] {bi/total_batches:6.2%} | {elapsed/60:.1f} min elapsed | ETA ~{eta/60:.1f} min", flush=True)

# ========= Save =========
for col, vals in z_cols.items():
    df[col] = vals

# Legacy aliases for downstream compatibility
df["z_greenwashing"]       = df["z_greenwashing_claims"]
df["z_transformativeness"] = df["z_transformative_action"]

df.to_csv(OUT_CSV, index=False, quoting=csv.QUOTE_MINIMAL)
print("Saved:", OUT_CSV, flush=True)
print(df[list(COLUMN_MAP.values()) + ["z_greenwashing", "z_transformativeness"]].head().to_string(index=False), flush=True)


Loaded 1,000 rows from /projappl/project_2004147/visions/bertopic_with_zeroshot_v9/mocked_dataset_generated_by_chatgpt.csv


/scratch/project_2004147/visions/sitepkgs/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/scratch/project_2004147/visions/sitepkgs/transformers/utils/hub.py:127: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


GPU available: True
Using GPU: Tesla V100-SXM2-32GB
BATCH_SIZE = 192
HuggingFace 'datasets' not available -> using manual batching.
[6/6] 100.00% | 1.4 min elapsed | ETA ~0.0 min
Saved: /projappl/project_2004147/visions/bertopic_with_zeroshot_v9/nff_transf_green_zeroshot_anon.csv
 z_nature  z_society  z_culture  z_greenwashing_discourse  z_greenwashing_claims  z_transformative_discourse  z_transformative_action  z_greenwashing  z_transformativeness
 0.942383   0.327393   0.105774                  0.003878               0.018036                    0.984863                 0.908203        0.018036              0.908203
 0.800781   0.865723   0.040192                  0.003216               0.007195                    0.681641                 0.982422        0.007195              0.982422
 0.990234   0.460449   0.551270                  0.007668               0.034088                    0.968262                 0.952637        0.034088              0.952637
 0.707520   0.046661   0.805664